In [1]:
from Run_Designs import design_experiments, design_initial_experiments
from methods.DataPrep import dataprep
import numpy as np
import pandas as pd
import logging
import time
import os
import pickle
import math

import matplotlib.pyplot as plt

Design Variable

1. C - List of number of categroies corresponding to each categorical variable
2. Nc - Number of categorical variables
3. Nx - Continuous variables
4. h - categorical
5. x - continuous
6. z - [h, x]
7. nDim - Nc + Nx
8. bounds - Lower and Upper bounds of both Catergorical variables and continuous variables
9. Var_unsc - Variables that shouldn't be scaled in the model

Sample Collection Details

10. n_iter - Number of iterations to run the algorithm
11. initN - Number of intial data points
12. batch_size (b) - Number of experiments to be generated in each iteration

Optimization Problem details
13. approach_type (type of design space)
    - Single system/task learning - CoCa (Continuous Categorical) vs Co (Continuous)
    - Multi system/task learning - CoCa_ICM, CoCa_LCM, CoCa_LEK, CoCa_LEK_precompute, 
    Co_ICM, Co_LCM, Co_LEK, Co_LEK_precompute
    
14. prob_type - 'UnConstrained' vs 'Constrained' (type of optimization problem)
15. method - 'thompson sampling' vs 'constant_liar' (type of sampling)
16. Meas_Noise - Measurement Noise in the target 
17. trade_off : Explorarion exploitation balance

Multitask/TL details

17. Noutput : Number of different systems in the data. Needed for multi-task or transfer learning application (System ID is always in the last column of the dataset)
18. if approach type for multi-task is ICM or LCM --> W_rank is a required parameter for the model. if approach type is LEK based then N_latent is required. Default W_rank = 1. Default N_latent = 2
19.



In [2]:
Stock_solid = pd.read_excel('CarbonSourceInfo.xlsx', 'Stocks_solid')
Stock_liquid = pd.read_excel('CarbonSourceInfo.xlsx', 'Stocks_liquid')

Carbon_Names = Stock_solid['Carbon Source'].values.tolist()
Carbon_Names.append(Stock_liquid['Carbon Source'][1])
Carbon_Names.append(Stock_liquid['Carbon Source'][2])

Carbon_Ub = [50]*17 #[g/L]
Carbon_Ub.append(10) # Glycerol [mL/L]
Carbon_Ub.append(50) #Ethanol [mL/L]

In [3]:
Mol_List = ['KPhaffii_HSA'] #, 'KPhaffii_RBDJ', 'Kphaffii_Trastuzumab'
Data_Molecules = {}
for m in Mol_List:
    Data_Molecules[m] = pd.read_excel("KPhaffii_Experiments.xlsx", m).iloc[:,1:]
    for i in range(Data_Molecules[m].shape[0]):
        Data_Molecules[m].iloc[i,0] = Carbon_Names.index(Data_Molecules[m].iloc[i,0])
        Data_Molecules[m].iloc[i,1] = Data_Molecules[m].iloc[i,1]/Carbon_Ub[Data_Molecules[m].iloc[i,0]]
        Data_Molecules[m].iloc[i,2:4] = Data_Molecules[m].iloc[i,2:4]/10
        
data_modified_np = {}
result_modified_np = {}
cnt_m = 0
for m in Mol_List:
    data_modified = []
    result_modified = []
    data_init = Data_Molecules[m].iloc[:,0:4].values.astype('float64')
    result_init = Data_Molecules[m]['SP [mg/L/OD600]'].values.reshape(-1,1).astype('float64')
    for nc in range(data_init.shape[0]):
        data_modified.append(data_init[nc, :])
        result_modified.append(result_init[nc])
    
    temp1 = np.array(data_modified)
    data_modified_np[m] = np.concatenate((temp1, cnt_m*np.ones((temp1.shape[0], 1))), axis =1)
    result_modified_np[m] = np.array(result_modified).reshape(-1,1)
    
    cnt_m = cnt_m+1

In [4]:
data = data_modified_np['KPhaffii_HSA']
result = result_modified_np['KPhaffii_HSA']

print(data.shape)
Nround = 5
path = '../Exp/BSA/'


for r in range(Nround):
    data_modified = []
    result_modified = []
    dat_read = pd.read_csv("./Round"+str(r)+'/Reconstructed_Round'+str(r)+"_CoCaLEK.csv").iloc[:,1:].values #
    result_new = pd.read_csv(path+"Round" +str(r)+"_CoCaLEK/Round" + str(r) +"_Result_Summary_final.csv")['Specific Productivity'].values.reshape(-1,1)
    for nc in range(dat_read.shape[0]):
        data_modified.append(dat_read[nc, :])
        result_modified.append(result_new[nc])
    
    temp1 = np.array(data_modified)
    data_new = np.concatenate((temp1, 1*np.ones((temp1.shape[0], 1))), axis =1)
    result_new_np = np.array(result_modified).reshape(-1,1)
    
    
    data = np.concatenate((data, data_new), axis = 0)
    result = np.concatenate((result, result_new_np), axis = 0)

(84, 5)


In [5]:
print(data.shape)

(139, 5)


In [6]:
import pickle
fnl = "./Round4/4_ModelBackground_CoCaLEK.pkl" #_updated
with open(fnl, "rb") as fp:
    ModelBackground = pickle.load(fp)
    
key = 'CoCa_LEK'
data_param_CoCaLEK = ModelBackground['data_param']
data_mod = dataprep(data, data_param_CoCaLEK, key)

In [7]:
# ModelBackground['data_param']['Nlatent'] = 1
# fnl = "./Round1/1_ModelBackground_CoCaLEK_updated.pkl" #

# with open(fnl, "wb") as fp:
#     pickle.dump(ModelBackground, fp)

In [8]:
method = 'constant_liar'
batch_size = 11
output_file_name = './Round5/Round5_CoCaLEK'
background_model_parameter = './Round4/4_ModelBackground_CoCaLEK.pkl' #_updated
new_model_parameter = './Round5/5_ModelBackground_CoCaLEK.pkl'

z_next, gp_actual = design_experiments(data_mod, result,batch_size, method,background_model_parameter, 
                            new_model_parameter , output_file_name)

[[10]
 [ 7]
 [11]
 [ 0]
 [ 0]
 [ 0]
 [ 0]
 [12]
 [ 7]
 [ 0]
 [15]]
[array([34.9885053 ,  1.        ,  1.        ,  1.        ,  1.        ,
        1.        ,  1.        ,  3.22083297,  1.        ,  1.        ,
        1.81555301,  1.55907766,  1.74415978,  1.        ,  1.        ,
        1.71628954,  1.        ,  1.        ,  1.        ])]
I am in the except zone
  index  |  GP_regression.mul.latentVariable.Wembedding  |  constraints  |  priors
  [0 0]  |                                   1.76405235  |               |        
  [0 1]  |                                   0.40015721  |               |        
  [1 0]  |                                   0.97873798  |               |        
  [1 1]  |                                   2.24089320  |               |        
  index  |  GP_regression.mul.latentVariable.Wembedding  |  constraints  |  priors
  [0 0]  |                                   1.47063922  |               |        
  [0 1]  |                                   1.087

In [ ]:
Carbon_Names[15]

In [ ]:
Carbon_Names[16]

In [ ]:
Carbon_Names[11]

In [ ]:
Carbon_Names[5]